In [ ]:
import os, sys, json
from pathlib import Path
from types import SimpleNamespace
runners = list(Path("/kaggle/input").rglob("phase5_b1_train.py"))
print("Mounted runners:", [str(p) for p in runners], flush=True)
if len(runners) != 1: raise RuntimeError("Expected one official-data runner")
input_dir = str(runners[0].parent)
sys.path[:0] = [input_dir + "/phase1_source", input_dir]
import phase5_b1_train as runner
"""Layer-wise partial B1 adaptation: only encoder layers 5â€“6 and head train."""
B1Classifier = runner.B1Classifier


class PartialB1Classifier(B1Classifier):
    def __init__(self, checkpoint):
        super().__init__(checkpoint)
        for block in self.backbone.encoder:
            block.attn.wqkv = block.attn.wqkv.base
            block.attn.wo = block.attn.wo.base
            for name in ("w1", "w2", "w3"):
                setattr(block.ffn, name, getattr(block.ffn, name).base)
        for param in self.parameters():
            param.requires_grad_(False)
        for block in self.backbone.encoder[-2:]:
            for param in block.parameters():
                param.requires_grad_(True)
        for param in self.head.parameters():
            param.requires_grad_(True)

runner.B1Classifier = PartialB1Classifier
output_dir = Path("/kaggle/working/partial_layers")
args = SimpleNamespace(input=input_dir, output=str(output_dir), epochs=2, batch=8, eval_batch=16, accum=2, lr=0.0001)
runner.train(args)
report_file = output_dir / "phase5_b1_lora_report.json"
report = json.loads(report_file.read_text())
report["method"] = "partial layer-wise fine-tuning of B1 encoder layers 5 and 6"
(output_dir / "phase5_b1_partial_report.json").write_text(json.dumps(report, indent=2))
(output_dir / "phase5_b1_lora_prediction.zip").rename(output_dir / "phase5_b1_partial_prediction.zip")
